In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Configuration

The path defaults work when the notebook is run from this repository. Set `HADL_DATA_PATH`, `HADL_EMBEDDINGS_PATH`, or `HADL_OUTPUT_DIR`, or edit the paths below when using Drive/Colab.


In [ ]:
from pathlib import Path
import os
import random
import json
import math

import numpy as np
import pandas as pd
import torch

from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from tqdm.auto import tqdm

SEED = 42
VALIDATION_GROUP_FRACTION = 0.15
LANDMARK_FRACTION = 0.05
FINAL_SAMPLE_FRACTION = 0.20

MODEL_NAME = "Qwen/Qwen3-1.7B"
MAX_PROMPT_TOKENS = 768
MAX_ANSWER_TOKENS = 256
LOSS_BATCH_SIZE = 1

KRR_ALPHA = 1e-2
KRR_NYSTROEM_COMPONENTS = 2048
KRR_GAMMA_SAMPLE_SIZE = 500
LAMBDA_RESIDUAL = 1.0
PRED_BATCH_SIZE = 512
PCA_RETAINED_VARIANCE = 0.80

TRAIN_EPOCHS = 2
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4


def find_project_root(start):
    for parent in (start, *start.parents):
        for candidate in (parent, parent / "hadl_courseproject"):
            if (candidate / "data_and_scripts/odia_synth_cleaned/or_synthetic_filtered.parquet").is_file():
                return candidate
    return None


PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None:
    default_data_path = Path("/content/drive/MyDrive/odia_embeddings/or_synthetic_filtered.parquet")
    default_embeddings_path = Path("/content/drive/MyDrive/odia_embeddings/odia_embeddings.pt")
    default_output_dir = Path("/content/drive/MyDrive/odia_embeddings")
else:
    default_data_path = PROJECT_ROOT / "data_and_scripts/odia_synth_cleaned/or_synthetic_filtered.parquet"
    default_embeddings_path = PROJECT_ROOT / "embedding_generation/odia_embeddings.pt"
    default_output_dir = PROJECT_ROOT / "low_rank_method"

DATA_PATH = Path(os.environ.get("HADL_DATA_PATH", str(default_data_path)))
EMBEDDINGS_PATH = Path(os.environ.get("HADL_EMBEDDINGS_PATH", str(default_embeddings_path)))
WORK_DIR = Path(os.environ.get("HADL_OUTPUT_DIR", str(default_output_dir)))
WORK_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Data:", DATA_PATH)
print("Embeddings:", EMBEDDINGS_PATH)
print("Output directory:", WORK_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Data: /content/drive/MyDrive/odia_embeddings/or_synthetic_filtered.parquet
Embeddings: /content/drive/MyDrive/odia_embeddings/odia_embeddings.pt
Output directory: /content/drive/MyDrive/odia_embeddings
CUDA available: True
GPU: Tesla T4


## 2. Load the full data and aligned embedding tensor

Do not sort, filter, or drop rows before assigning source indices. The existing embedding notebook saves a CPU tensor with one normalized embedding per source row.

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")
if not EMBEDDINGS_PATH.is_file():
    raise FileNotFoundError(f"Embedding tensor not found: {EMBEDDINGS_PATH}")

required_columns = ["paragraph", "question", "answer", "wiki_id"]
df = pd.read_parquet(DATA_PATH).reset_index(drop=True)
missing = [column for column in required_columns if column not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")
if df["wiki_id"].isna().any():
    raise ValueError("wiki_id contains missing values; every row must belong to a group")
for column in ["paragraph", "question", "answer"]:
    if df[column].isna().any():
        raise ValueError(f"{column} contains missing values")
    df[column] = df[column].astype(str)
    if df[column].str.strip().eq("").any():
        raise ValueError(f"{column} contains blank values")

df["_source_row_idx"] = np.arange(len(df), dtype=np.int64)
df["_wiki_group"] = df["wiki_id"].astype(str)

try:
    embedding_tensor = torch.load(EMBEDDINGS_PATH, map_location="cpu", weights_only=True)
except TypeError:  # compatibility with older PyTorch releases
    embedding_tensor = torch.load(EMBEDDINGS_PATH, map_location="cpu")
if not isinstance(embedding_tensor, torch.Tensor):
    raise TypeError("Expected odia_embeddings.pt to contain a single torch.Tensor")
X_full = embedding_tensor.detach().cpu().numpy().astype(np.float32, copy=False)
del embedding_tensor

if X_full.ndim != 2:
    raise ValueError(f"Expected a 2D embedding matrix; got shape {X_full.shape}")
if X_full.shape[0] != len(df):
    raise ValueError(
        f"Embedding rows ({X_full.shape[0]}) != dataset rows ({len(df)}). "
        "Restore exact source order or regenerate embeddings."
    )
if not np.isfinite(X_full).all():
    raise ValueError("Embeddings contain NaN or infinite values")
row_norms = np.linalg.norm(X_full, axis=1)
if (row_norms <= 1e-12).any():
    raise ValueError("At least one embedding has zero norm")
X_full /= row_norms[:, None]

print("Rows:", len(df))
print("Embedding shape:", X_full.shape)
print("Unique wiki_id groups:", df["_wiki_group"].nunique())

Rows: 216394
Embedding shape: (216394, 768)
Unique wiki_id groups: 18287


## 3. Make the group-aware train/validation split first

The split is 85/15 by `wiki_id`, with seed 42. GroupShuffleSplit assigns whole groups; row counts may differ slightly from 85/15 because wiki groups have different numbers of QA rows. Keep source order within each saved split.

In [ ]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=VALIDATION_GROUP_FRACTION,
    random_state=SEED,
)
train_source_idx, validation_source_idx = next(
    group_splitter.split(df, groups=df["_wiki_group"])
)
train_source_idx = np.sort(train_source_idx)
validation_source_idx = np.sort(validation_source_idx)

train_df = df.iloc[train_source_idx].reset_index(drop=True).copy()
validation_df = df.iloc[validation_source_idx].reset_index(drop=True).copy()
train_groups = set(train_df["_wiki_group"])
validation_groups = set(validation_df["_wiki_group"])
if train_groups & validation_groups:
    raise AssertionError("wiki_id group leakage between train and validation")
if len(train_df) + len(validation_df) != len(df):
    raise AssertionError("Train and validation rows do not cover the source data")

X_train = np.ascontiguousarray(X_full[train_source_idx], dtype=np.float32)
del X_full

train_path = WORK_DIR / "landmark_losses/train_85_percent.parquet"
validation_path = WORK_DIR / "landmark_losses/validation_15_percent.parquet"
train_df.to_parquet(train_path, index=False)
validation_df.to_parquet(validation_path, index=False)

print(f"Train rows: {len(train_df):,}; groups: {len(train_groups):,}")
print(f"Validation rows: {len(validation_df):,}; groups: {len(validation_groups):,}")
print("Saved:", train_path)
print("Saved:", validation_path)

Train rows: 185,290; groups: 15,543
Validation rows: 31,104; groups: 2,744
Saved: /content/drive/MyDrive/odia_embeddings/train_85_percent.parquet
Saved: /content/drive/MyDrive/odia_embeddings/validation_15_percent.parquet


## 4. Compute leverage scores on train rows and save both partitions

For a thin QR basis `Q` of the training embedding matrix, each row leverage score is the squared row norm of `Q` (restricted to its numerical rank). Fixed-size PPS inclusion probabilities are capped at one and sum to the requested sample size. Systematic PPS sampling then returns exactly that many unique training rows.

In [ ]:
def compute_row_leverage_scores(X):
    Q, R = np.linalg.qr(X, mode="reduced")
    U, singular_values, _ = np.linalg.svd(R, full_matrices=False)
    if len(singular_values) == 0 or singular_values[0] <= 0:
        raise ValueError("Embedding matrix has zero numerical rank")
    tolerance = max(X.shape) * np.finfo(singular_values.dtype).eps * singular_values[0]
    rank = int(np.sum(singular_values > tolerance))
    if rank == 0:
        raise ValueError("Embedding matrix has zero numerical rank")
    Q_rank = Q @ U[:, :rank]
    scores = np.sum(Q_rank * Q_rank, axis=1, dtype=np.float64)
    return scores, rank


def fixed_size_pps_inclusion_probabilities(weights, sample_size):
    weights = np.asarray(weights, dtype=np.float64)
    if weights.ndim != 1 or not np.isfinite(weights).all() or (weights < 0).any():
        raise ValueError("PPS weights must be a finite, nonnegative vector")
    active = np.flatnonzero(weights > 0)
    if len(active) < sample_size:
        raise ValueError("Fewer positive leverage scores than requested sample rows")

    inclusion = np.zeros(len(weights), dtype=np.float64)
    remaining = float(sample_size)
    while len(active) and remaining > 1e-12:
        scale = remaining / weights[active].sum()
        saturated_mask = weights[active] * scale >= 1.0
        if not saturated_mask.any():
            inclusion[active] = weights[active] * scale
            remaining = 0.0
            break
        saturated = active[saturated_mask]
        inclusion[saturated] = 1.0
        remaining -= len(saturated)
        active = active[~saturated_mask]

    delta = sample_size - inclusion.sum()
    if abs(delta) > 1e-8:
        fractional = np.flatnonzero((inclusion > 0) & (inclusion < 1))
        if not len(fractional):
            raise ArithmeticError("Could not normalize fixed-size PPS probabilities")
        inclusion[fractional[-1]] += delta
    if (inclusion < -1e-10).any() or (inclusion > 1 + 1e-10).any():
        raise ArithmeticError("Invalid PPS inclusion probabilities")
    return np.clip(inclusion, 0.0, 1.0)


def systematic_pps_sample(inclusion_probabilities, sample_size, seed):
    cumulative = np.cumsum(inclusion_probabilities, dtype=np.float64)
    cumulative[-1] = float(sample_size)
    points = np.random.default_rng(seed).random() + np.arange(sample_size)
    selected = np.searchsorted(cumulative, points, side="right")
    if (selected >= len(inclusion_probabilities)).any():
        raise ArithmeticError("Systematic PPS sampler produced an out-of-range row")
    if len(np.unique(selected)) != sample_size:
        raise ArithmeticError("Systematic PPS sampler did not return unique rows")
    return selected


leverage_scores, embedding_rank = compute_row_leverage_scores(X_train)
landmark_count = max(1, int(round(LANDMARK_FRACTION * len(train_df))))
if landmark_count >= len(train_df):
    raise ValueError("Landmark fraction leaves no remaining training rows")
landmark_inclusion_probs = fixed_size_pps_inclusion_probabilities(
    leverage_scores, landmark_count
)
landmark_train_idx = np.sort(
    systematic_pps_sample(landmark_inclusion_probs, landmark_count, SEED)
)
landmark_mask = np.zeros(len(train_df), dtype=bool)
landmark_mask[landmark_train_idx] = True

selected_df = train_df.iloc[landmark_train_idx].copy().reset_index(drop=True)
selected_df["leverage_score"] = leverage_scores[landmark_train_idx]
selected_df["inclusion_probability"] = landmark_inclusion_probs[landmark_train_idx]
remaining_df = train_df.loc[~landmark_mask].copy().reset_index(drop=True)

selected_path = WORK_DIR / "landmark_losses/selected_05_percent.parquet"
remaining_path = WORK_DIR / "landmark_losses/remaining_95_percent.parquet"
selected_df.to_parquet(selected_path, index=False)
remaining_df.to_parquet(remaining_path, index=False)

selection_audit = train_df[["_source_row_idx", "wiki_id"]].copy()
selection_audit["leverage_score"] = leverage_scores
selection_audit["inclusion_probability"] = landmark_inclusion_probs
selection_audit["is_landmark"] = landmark_mask
selection_audit.to_parquet(WORK_DIR / "landmark_losses/leverage_selection_audit.parquet", index=False)

if not np.isclose(leverage_scores.sum(), embedding_rank, rtol=1e-3, atol=1e-3):
    raise AssertionError("Leverage scores do not sum to the numerical embedding rank")
if len(selected_df) + len(remaining_df) != len(train_df):
    raise AssertionError("Landmark and remaining partitions do not cover train rows")
if set(selected_df["_source_row_idx"]) & set(remaining_df["_source_row_idx"]):
    raise AssertionError("Landmark and remaining partitions overlap")

print("Numerical embedding rank:", embedding_rank)
print("Leverage score sum:", float(leverage_scores.sum()))
print(f"Saved landmarks: {len(selected_df):,} ({selected_path})")
print(f"Saved remaining train rows: {len(remaining_df):,} ({remaining_path})")

Numerical embedding rank: 141
Leverage score sum: 141.00000007482944
Saved landmarks: 9,264 (/content/drive/MyDrive/odia_embeddings/selected_05_percent.parquet)
Saved remaining train rows: 176,026 (/content/drive/MyDrive/odia_embeddings/remaining_95_percent.parquet)


## 5. Load the current Qwen3-1.7B model

These forward passes score only the selected 20% training landmarks. The model remains unfine-tuned during loss estimation.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before loading Qwen3-1.7B")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch.nn.functional as F

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)
model.eval()
model.config.use_cache = False
print("Loaded:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute dtype:", compute_dtype)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loaded: Qwen/Qwen3-1.7B
GPU: Tesla T4
Compute dtype: torch.bfloat16


## 6. Shared Odia prompt and answer-only landmark losses

The same prompt builder and mean answer-token loss are used for current-model scoring and QLoRA training. Prompt tokens are masked with `-100`; answer tokens and the end-of-sequence token are supervised.

In [ ]:
SYSTEM_INSTRUCTION = (
    "ତୁମେ ଜଣେ ଓଡ଼ିଆ ପ୍ରଶ୍ନ-ଉତ୍ତର ସହାୟକ। "
    "ଦିଆଯାଇଥିବା ଅନୁଚ୍ଛେଦ ଆଧାରରେ ପ୍ରଶ୍ନର ଉତ୍ତର "
    "ଓଡ଼ିଆ ଭାଷା ଓ ଓଡ଼ିଆ ଲିପିରେ ଦିଅ। "
    "ଅନୁଚ୍ଛେଦରେ ଉତ୍ତର ନଥିଲେ ଅନୁମାନ କରନାହିଁ।"
)


def build_prompt(row):
    get = (lambda key: row[key]) if isinstance(row, dict) else (lambda key: row[key])
    return (
        f"{SYSTEM_INSTRUCTION}\n\n"
        f"ଅନୁଚ୍ଛେଦ:\n{str(get('paragraph'))}\n\n"
        f"ପ୍ରଶ୍ନ:\n{str(get('question'))}\n\n"
        "ଉତ୍ତର:\n"
    )


def build_example(row):
    prompt_ids = tokenizer(
        build_prompt(row),
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    )["input_ids"]
    answer_ids = tokenizer(
        str(row["answer"]),
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_ANSWER_TOKENS - 1,
    )["input_ids"]
    if tokenizer.eos_token_id is not None:
        answer_ids = answer_ids + [tokenizer.eos_token_id]
    if not answer_ids:
        raise ValueError("An example has no supervised answer tokens")
    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids
    return input_ids, labels


@torch.inference_mode()
def compute_landmark_losses(frame, landmark_positions, batch_size=None,
                            max_batch=2, max_tokens=6144, save_every=50):
    ckpt = WORK_DIR / "landmark_losses/landmark_losses_partial.npy"
    rows = frame.iloc[landmark_positions].to_dict(orient="records")
    encoded = [build_example(r) for r in rows]
    lengths = np.array([len(ids) for ids, _ in encoded])
    losses = np.load(ckpt) if ckpt.exists() else np.full(len(rows), np.nan, dtype=np.float32)
    todo = [i for i in np.argsort(lengths) if np.isnan(losses[i])]   # short -> long
    print(f"{len(rows) - len(todo)} already done, {len(todo)} to score")

    # greedy batches under a padded-token budget
    batches, cur = [], []
    for i in todo:
        if cur and (len(cur) >= max_batch or (len(cur) + 1) * lengths[i] > max_tokens):
            batches.append(cur); cur = []
        cur.append(i)
    if cur:
        batches.append(cur)

    model.eval()
    pad = tokenizer.pad_token_id
    for b, idxs in enumerate(tqdm(batches, desc="Landmark losses")):
        L = max(lengths[i] for i in idxs)
        ids = torch.tensor([encoded[i][0] + [pad] * (L - lengths[i]) for i in idxs], device="cuda")
        lab = torch.tensor([encoded[i][1] + [-100] * (L - lengths[i]) for i in idxs], device="cuda")
        att = torch.tensor([[1] * lengths[i] + [0] * (L - lengths[i]) for i in idxs], device="cuda")

        hidden = model.model(input_ids=ids, attention_mask=att).last_hidden_state[:, :-1]
        tgt = lab[:, 1:]
        m = tgt != -100
        logits = model.lm_head(hidden[m]).float()                    # only answer positions
        nll = F.cross_entropy(logits, tgt[m], reduction="none")
        row = torch.arange(len(idxs), device="cuda")[:, None].expand_as(tgt)[m]
        sums = torch.zeros(len(idxs), device="cuda").index_add_(0, row, nll)
        per_example = (sums / m.sum(1)).cpu().numpy()
        for j, i in enumerate(idxs):
            losses[i] = per_example[j]
        if b % save_every == 0:
            np.save(ckpt, losses)
    np.save(ckpt, losses)
    assert np.isfinite(losses).all()
    return losses

landmark_losses = compute_landmark_losses(train_df, landmark_train_idx, LOSS_BATCH_SIZE)
np.save(WORK_DIR / "landmark_losses/landmark_losses.npy", landmark_losses)
answer_token_lengths = np.asarray([
    len(tokenizer(
        str(answer),
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_ANSWER_TOKENS - 1,
    )["input_ids"]) + (1 if tokenizer.eos_token_id is not None else 0)
    for answer in selected_df["answer"]
], dtype=np.int32)
landmark_loss_table = selected_df[["_source_row_idx", "wiki_id"]].copy().reset_index(drop=True)
landmark_loss_table["answer_token_length"] = answer_token_lengths
landmark_loss_table["true_answer_mean_nll"] = landmark_losses
landmark_loss_table.to_parquet(WORK_DIR / "landmark_losses/landmark_losses.parquet", index=False)

if len(landmark_losses) != len(selected_df):
    raise AssertionError("Landmark loss vector is not aligned with the selected parquet")
loss_length_corr = (
    float(np.corrcoef(landmark_losses, answer_token_lengths)[0, 1])
    if np.std(landmark_losses) > 0 and np.std(answer_token_lengths) > 0
    else float("nan")
)
print({
    "n": len(landmark_losses),
    "mean": float(landmark_losses.mean()),
    "std": float(landmark_losses.std()),
    "min": float(landmark_losses.min()),
    "max": float(landmark_losses.max()),
    "loss_vs_answer_token_length_correlation": loss_length_corr,
})

6310 already done, 2954 to score


Landmark losses:   0%|          | 0/1477 [00:00<?, ?it/s]

{'n': 9264, 'mean': 2.643186569213867, 'std': 1.9034161567687988, 'min': 0.16542287170886993, 'max': 16.752971649169922, 'loss_vs_answer_token_length_correlation': -0.5241980430089718}


## 7. Fit scalable RBF KRR and predict train-row losses

A dense exact RBF kernel for tens of thousands of landmarks is too large for a typical notebook runtime. Nyström features approximate the RBF kernel while using every landmark loss as a regression target. The held-out-landmark diagnostic is split only from train landmarks; validation groups remain untouched.

In [ ]:
X_landmarks = X_train[landmark_train_idx]
rng = np.random.default_rng(SEED)
probe_n = min(KRR_GAMMA_SAMPLE_SIZE, len(X_landmarks))
probe = X_landmarks[rng.choice(len(X_landmarks), size=probe_n, replace=False)]
distance_sq = euclidean_distances(probe, squared=True)
upper = distance_sq[np.triu_indices_from(distance_sq, k=1)]
positive_distance_sq = upper[upper > 1e-12]
gamma = 1.0 / np.median(positive_distance_sq) if len(positive_distance_sq) else 1.0

landmark_positions = np.arange(len(landmark_train_idx))
fit_lm, heldout_lm = train_test_split(
    landmark_positions,
    test_size=0.20,
    random_state=SEED,
)
cv_components = min(KRR_NYSTROEM_COMPONENTS, len(fit_lm))
cv_mapper = Nystroem(
    kernel="rbf",
    gamma=gamma,
    n_components=cv_components,
    random_state=SEED,
)
fit_features = cv_mapper.fit_transform(X_landmarks[fit_lm])
cv_regressor = Ridge(alpha=KRR_ALPHA, fit_intercept=False)
cv_regressor.fit(fit_features, landmark_losses[fit_lm])
heldout_prediction = cv_regressor.predict(cv_mapper.transform(X_landmarks[heldout_lm]))
print("Held-out landmark MAE:", mean_absolute_error(landmark_losses[heldout_lm], heldout_prediction))
print("Held-out landmark RMSE:", mean_squared_error(landmark_losses[heldout_lm], heldout_prediction) ** 0.5)

del fit_features, cv_mapper, cv_regressor, heldout_prediction
nystroem_components = min(KRR_NYSTROEM_COMPONENTS, len(landmark_train_idx))
feature_map = Nystroem(
    kernel="rbf",
    gamma=gamma,
    n_components=nystroem_components,
    random_state=SEED,
)
landmark_features = feature_map.fit_transform(X_landmarks)
krr_regressor = Ridge(alpha=KRR_ALPHA, fit_intercept=False)
krr_regressor.fit(landmark_features, landmark_losses)
del landmark_features

predicted_losses = np.empty(len(train_df), dtype=np.float32)
for start in tqdm(range(0, len(train_df), PRED_BATCH_SIZE), desc="Train loss predictions"):
    end = min(start + PRED_BATCH_SIZE, len(train_df))
    features = feature_map.transform(X_train[start:end])
    predicted_losses[start:end] = krr_regressor.predict(features).astype(np.float32)
    del features
predicted_losses[landmark_train_idx] = landmark_losses
np.save(WORK_DIR / "residual_computations/predicted_train_losses.npy", predicted_losses)
if not np.isfinite(predicted_losses).all():
    raise ValueError("Predicted losses contain NaN or infinity")
print("RBF gamma:", gamma, "ridge alpha:", KRR_ALPHA)
print("Nyström components:", nystroem_components)
print("Predicted train loss mean/std:", float(predicted_losses.mean()), float(predicted_losses.std()))

Held-out landmark MAE: 1.0966976881027222
Held-out landmark RMSE: 1.5547734529131318


Train loss predictions:   0%|          | 0/362 [00:00<?, ?it/s]

RBF gamma: 2.338789 ridge alpha: 0.01
Nyström components: 2048
Predicted train loss mean/std: 2.643064260482788 1.1623709201812744


## 8. Residual distance from the 80%-variance landmark PCA subspace

The 80% PCA choice is intentional: residual energy measures how far rows lie in lower-variance directions outside the retained landmark subspace.

In [ ]:
_, singular_values, Vt = np.linalg.svd(X_landmarks, full_matrices=False)
explained_variance = singular_values ** 2
explained_variance_ratio = explained_variance / explained_variance.sum()
cumulative_variance = np.cumsum(explained_variance_ratio)
pca_components = int(np.searchsorted(cumulative_variance, PCA_RETAINED_VARIANCE) + 1)
basis = Vt[:pca_components].T.astype(np.float32, copy=False)
retained_variance = float(cumulative_variance[pca_components - 1])

residual_sq = np.empty(len(train_df), dtype=np.float32)
for start in tqdm(range(0, len(train_df), PRED_BATCH_SIZE), desc="PCA residuals"):
    end = min(start + PRED_BATCH_SIZE, len(train_df))
    batch = X_train[start:end]
    projected = (batch @ basis) @ basis.T
    residual_sq[start:end] = np.sum((batch - projected) ** 2, axis=1)
residual_sq = np.maximum(residual_sq, 0.0)
np.save(WORK_DIR / "residual_computations/train_residual_sq.npy", residual_sq)
if not np.isfinite(residual_sq).all():
    raise ValueError("Residuals contain NaN or infinity")
print("PCA components:", pca_components)
print("Retained cumulative variance:", retained_variance)
print("Residual mean/max:", float(residual_sq.mean()), float(residual_sq.max()))

PCA residuals:   0%|          | 0/362 [00:00<?, ?it/s]

PCA components: 3
Retained cumulative variance: 0.8062711358070374
Residual mean/max: 0.19023846089839935 0.3046298623085022


## 9. Scores, weighted draws, and audit outputs

The final sample is drawn with replacement from train rows. Duplicate draws remain separate training examples. Each draw stores the requested `1/(s*p)` weight; the Trainer uses the equivalent `1/(N*p)` per-draw weight for a minibatch-mean objective.

In [ ]:
scores = np.maximum(predicted_losses, 0.0) + LAMBDA_RESIDUAL * residual_sq
if not np.isfinite(scores).all():
    raise ValueError("Scores contain NaN or infinity")
if scores.sum() <= 0:
    raise ValueError("All scores are zero; revise the score configuration")

probabilities = scores / scores.sum()
N_train = len(train_df)
num_draws = max(1, int(round(FINAL_SAMPLE_FRACTION * N_train)))
if not np.isclose(probabilities.sum(), 1.0):
    raise AssertionError("Sampling probabilities do not sum to one")

draw_rng = np.random.default_rng(SEED + 1)
draw_train_idx = draw_rng.choice(
    N_train,
    size=num_draws,
    replace=True,
    p=probabilities,
)
draw_probability = probabilities[draw_train_idx]
final_df = train_df.iloc[draw_train_idx].copy().reset_index(drop=True)
final_df["draw_id"] = np.arange(num_draws, dtype=np.int64)
final_df["predicted_loss"] = predicted_losses[draw_train_idx]
final_df["residual_sq"] = residual_sq[draw_train_idx]
final_df["sampling_probability"] = draw_probability
final_df["draw_weight_1_over_s_p"] = 1.0 / (num_draws * draw_probability)
final_df["minibatch_importance_weight"] = 1.0 / (N_train * draw_probability)

final_draws_path = WORK_DIR / "odia_low_rank_run/final_weighted_train_draws.parquet"
final_df.to_parquet(final_draws_path, index=False)
np.save(WORK_DIR / "residual_computations/train_sampling_probabilities.npy", probabilities)

audit = train_df[["_source_row_idx", "wiki_id"]].copy()
audit["is_landmark"] = landmark_mask
audit["true_landmark_loss"] = np.nan
audit.loc[landmark_train_idx, "true_landmark_loss"] = landmark_losses
audit["predicted_loss"] = predicted_losses
audit["residual_sq"] = residual_sq
audit["score"] = scores
audit["sampling_probability"] = probabilities
audit.to_parquet(WORK_DIR / "residual_computations/train_selection_audit.parquet", index=False)

print("Saved weighted training draws:", final_draws_path)
print("Train rows / draws:", N_train, num_draws)

print("Weight min/mean/max:",
      float(final_df["draw_weight_1_over_s_p"].min()),
      float(final_df["draw_weight_1_over_s_p"].mean()),
      float(final_df["draw_weight_1_over_s_p"].max()))

Saved weighted training draws: /content/drive/MyDrive/odia_embeddings/final_weighted_train_draws.parquet
Train rows / draws: 185290 37058
Weight min/mean/max: 0.8788540959358215 4.999922752380371 105.77113342285156


In [ ]:
import torch
import pandas as pd

# Define paths
final_draws_path = WORK_DIR / "odia_low_rank_run/final_weighted_train_draws.parquet"
output_embeddings_path = WORK_DIR / "odia_low_rank_run/final_weighted_train_draws_embeddings.pt"

print(f"Loading selected draws from: {final_draws_path}")
final_df = pd.read_parquet(final_draws_path)

# Extract source row indices of the selected draws
selected_indices = final_df["_source_row_idx"].values

print(f"Loading original full embeddings from: {EMBEDDINGS_PATH}")
try:
    full_embeddings = torch.load(EMBEDDINGS_PATH, map_location="cpu", weights_only=True)
except TypeError:
    full_embeddings = torch.load(EMBEDDINGS_PATH, map_location="cpu")

# Slice the master tensor using the selected indices
print("Extracting corresponding embeddings...")
selected_embeddings = full_embeddings[selected_indices]

# Save the aligned embeddings
torch.save(selected_embeddings, output_embeddings_path)
print(f"Saved aligned embeddings to: {output_embeddings_path}")
print(f"Shape of final embeddings tensor: {selected_embeddings.shape}")

Loading selected draws from: /content/drive/MyDrive/odia_embeddings/final_weighted_train_draws.parquet
Loading original full embeddings from: /content/drive/MyDrive/odia_embeddings/odia_embeddings.pt
Extracting corresponding embeddings...
Saved aligned embeddings to: /content/drive/MyDrive/odia_embeddings/final_weighted_train_draws_embeddings.pt
Shape of final embeddings tensor: torch.Size([37058, 768])


In [ ]:
import os
import torch

output_embeddings_path = WORK_DIR / "odia_low_rank_run/final_weighted_train_draws_embeddings.pt"
if output_embeddings_path.exists():
    print(f"File exists: {output_embeddings_path}")
    print(f"File size: {os.path.getsize(output_embeddings_path) / (1024**2):.2f} MB")
    try:
        loaded_emb = torch.load(output_embeddings_path, map_location="cpu", weights_only=True)
    except TypeError:
        loaded_emb = torch.load(output_embeddings_path, map_location="cpu")
    print(f"Successfully loaded aligned embeddings tensor of shape: {loaded_emb.shape}")
else:
    print(f"Alining embeddings is still running or file not found at: {output_embeddings_path}")

File exists: /content/drive/MyDrive/odia_embeddings/final_weighted_train_draws_embeddings.pt
File size: 108.57 MB
Successfully loaded aligned embeddings tensor of shape: torch.Size([37058, 768])


## 10. Fine-tune on weighted train draws and evaluate on held-out groups

The group-held-out validation parquet is used for evaluation only. It is never included in leverage selection, loss estimation, KRR fitting, residual scoring, or training draws.

In [ ]:
from dataclasses import dataclass
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

# Release the scoring model, then reload the original pretrained checkpoint for QLoRA.
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


def encode_weighted_train_row(row):
    input_ids, labels = build_example(row)
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "importance_weight": float(row["minibatch_importance_weight"]),
    }


def encode_validation_row(row):
    input_ids, labels = build_example(row)
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


train_data = Dataset.from_pandas(final_df, preserve_index=False).map(
    encode_weighted_train_row,
    remove_columns=list(final_df.columns),
)
validation_data = Dataset.from_pandas(validation_df, preserve_index=False).map(
    encode_validation_row,
    remove_columns=list(validation_df.columns),
)


@dataclass
class WeightedCollator:
    tokenizer: object

    def __call__(self, features):
        max_len = max(len(feature["input_ids"]) for feature in features)
        pad_id = self.tokenizer.pad_token_id
        batch = {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }
        has_weights = "importance_weight" in features[0]
        if has_weights:
            weights = []
        for feature in features:
            pad_len = max_len - len(feature["input_ids"])
            batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad_len)
            batch["attention_mask"].append(feature["attention_mask"] + [0] * pad_len)
            batch["labels"].append(feature["labels"] + [-100] * pad_len)
            if has_weights:
                weights.append(feature["importance_weight"])
        tensors = {
            key: torch.tensor(value, dtype=torch.long)
            for key, value in batch.items()
        }
        if has_weights:
            tensors["importance_weight"] = torch.tensor(weights, dtype=torch.float32)
        return tensors


class ImportanceWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        weights = inputs.pop("importance_weight", None)
        labels = inputs["labels"]
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        )
        logits = outputs.logits[:, :-1, :].float()
        targets = labels[:, 1:]
        mask = targets != -100
        token_nll = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=-100,
            reduction="none",
        ).view(targets.shape)
        token_counts = mask.sum(dim=1).clamp(min=1)
        per_example_loss = (token_nll * mask).sum(dim=1) / token_counts
        if weights is None:
            loss = per_example_loss.mean()
        else:
            weights = weights.to(per_example_loss.device, dtype=per_example_loss.dtype)
            loss = (weights * per_example_loss).mean()
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir=str(WORK_DIR / "qwen3_odia_checkpoints"),
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=10,
    gradient_checkpointing=True,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)

trainer = ImportanceWeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=validation_data,
    data_collator=WeightedCollator(tokenizer),
)
trainer.train()

In [ ]:
ADAPTER_DIR = WORK_DIR / "qwen3_odia_lora_adapter"
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved LoRA adapter:", ADAPTER_DIR)

## 11. Validation-group inference smoke test

In [ ]:
model.eval()
model.config.use_cache = True
example = validation_df.iloc[0]
prompt = build_prompt(example)
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_PROMPT_TOKENS,
).to("cuda")
with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
new_tokens = generated[0, inputs["input_ids"].shape[1]:]
print("Question:", example["question"])
print("Reference:", example["answer"])
print("Prediction:", tokenizer.decode(new_tokens, skip_special_tokens=True))

## Pipeline checks

- The train/validation split is disjoint by `wiki_id`; only train rows reach leverage selection and downstream fitting.
- The 20% landmarks and remaining train rows are disjoint and cover train exactly.
- Landmark labels are answer-only mean token NLL, aligned with `selected_20_percent.parquet`.
- KRR predictions and PCA residuals are finite; residual PCA retains at least 80% cumulative variance.
- Train sampling probabilities sum to one, and final draws preserve duplicates and inverse-probability weights.
- The validation set is group-held-out. A separate test set and final benchmark should be created before reporting model quality.